In [3]:
import copy
from heapq import heappush, heappop

# Thay đổi n = 4 cho bài toán 15-puzzle
n = 4

# 4 vị trí dịch chuyển tương ứng bottom, left, top, right
rows = [ 1, 0, -1, 0 ]
cols = [ 0, -1, 0, 1 ]

# Tạo một lớp hàng đợi ưu tiên
class priorityQueue:
    def __init__(self):
        self.heap = []

    # Inserting a new key 'key'
    def push(self, key):
        heappush(self.heap, key)

    # Funct to remove the element that is min from the Priority Queue
    def pop(self):
        return heappop(self.heap)

    # Funct to check if the Queue is empty or not
    def empty(self):
        if not self.heap:
            return True
        else:
            return False

# Structure of the node
class nodes:
    def __init__(self, parent, mats, empty_tile_posi, costs, levels):
        self.parent = parent

        # Useful for storing the matrix
        self.mats = mats
        self.empty_tile_posi = empty_tile_posi
        self.costs = costs
        self.levels = levels

    def __lt__(self, nxt):
        return (self.costs + self.levels) < (nxt.costs + nxt.levels)

# Tính số ô bị sai vị trí (Misplaced tiles) - Theo đúng logic ảnh aa4e69
def calculateCosts(mats, final) -> int:
    count = 0
    for i in range(n):
        for j in range(n):
            if ((mats[i][j]) and (mats[i][j] != final[i][j])):
                count += 1
    return count

def newNodes(mats, empty_tile_posi, new_empty_tile_posi, levels, parent, final) -> nodes:
    # Copying data from the parent matrixes to the present matrixes
    new_mats = copy.deepcopy(mats)

    # Moving the tile by 1 position
    x1 = empty_tile_posi[0]
    y1 = empty_tile_posi[1]
    x2 = new_empty_tile_posi[0]
    y2 = new_empty_tile_posi[1]

    new_mats[x1][y1], new_mats[x2][y2] = new_mats[x2][y2], new_mats[x1][y1]

    # Setting the no. of misplaced tiles
    costs = calculateCosts(new_mats, final)

    new_nodes = nodes(parent, new_mats, new_empty_tile_posi, costs, levels)
    return new_nodes

# func to print the N by N matrix
def printMatsrix(mats):
    for i in range(n):
        for j in range(n):
            print("%d " % (mats[i][j]), end = " ")
        print()

# func to know if (x, y) is a valid or invalid matrix coordinates
def isSafe(x, y):
    return x >= 0 and x < n and y >= 0 and y < n

# Printing the path from the root node to the final node
def printPath(root):
    if root == None:
        return

    printPath(root.parent)
    printMatsrix(root.mats)
    print()

def solve(initial, empty_tile_posi, final):
    pq = priorityQueue()

    # Creating the root node
    costs = calculateCosts(initial, final)
    root = nodes(None, initial, empty_tile_posi, costs, 0)

    # Adding root to the list of live nodes
    pq.push(root)

    while not pq.empty():
        minimum = pq.pop()

        # If minimum.costs is 0, it means we reached the goal
        if minimum.costs == 0:
            printPath(minimum)
            return

        # Generating all feasible children
        for i in range(4):
            new_tile_posi = [
                minimum.empty_tile_posi[0] + rows[i],
                minimum.empty_tile_posi[1] + cols[i],
            ]

            if isSafe(new_tile_posi[0], new_tile_posi[1]):
                child = newNodes(minimum.mats,
                                 minimum.empty_tile_posi,
                                 new_tile_posi,
                                 minimum.levels + 1,
                                 minimum, final)
                pq.push(child)

# --- Thiết lập dữ liệu cho bài toán 15-puzzle ---

# Trạng thái bắt đầu (Ví dụ đơn giản để máy chạy nhanh)
initial = [ [ 1, 2, 3, 4 ],
            [ 5, 6, 7, 8 ],
            [ 9, 10, 0, 11 ],
            [ 13, 14, 15, 12 ] ]

# Trạng thái đích
final = [ [ 1, 2, 3, 4 ],
          [ 5, 6, 7, 8 ],
          [ 9, 10, 11, 12 ],
          [ 13, 14, 15, 0 ] ]

# Vị trí ô trống ban đầu (số 0) ở hàng 2, cột 2 (tính từ 0)
empty_tile_posi = [ 2, 2 ]

solve(initial, empty_tile_posi, final)

1  2  3  4  
5  6  7  8  
9  10  0  11  
13  14  15  12  

1  2  3  4  
5  6  7  8  
9  10  11  0  
13  14  15  12  

1  2  3  4  
5  6  7  8  
9  10  11  12  
13  14  15  0  



In [2]:
import heapq

def a_star_search(graph, heuristics, start, goal):
    """
    graph: Dictionary chứa kề và trọng số { 'A': [('B', 1), ('C', 3)], ... }
    heuristics: Dictionary chứa giá trị h(n) từ đỉnh n đến đích { 'A': 10, 'B': 8, ... }
    """

    # Priority Queue (tập OPEN): lưu (f_score, current_node)
    open_set = []
    heapq.heappush(open_set, (heuristics[start], start))

    # Lưu chi phí từ điểm xuất phát đến node hiện tại (g_score)
    g_score = {node: float('inf') for node in graph}
    g_score[start] = 0

    # Lưu vết đường đi
    came_from = {}

    # Tập CLOSED (để theo dõi các node đã được xử lý xong)
    closed_set = set()

    while open_set:
        # Lấy node có f_score nhỏ nhất
        current_f, current_node = heapq.heappop(open_set)

        # Nếu đã đến đích
        if current_node == goal:
            return reconstruct_path(came_from, current_node), g_score[goal]

        closed_set.add(current_node)

        # Duyệt các đỉnh kề
        for neighbor, weight in graph.get(current_node, []):
            if neighbor in closed_set:
                continue

            # g(n) tạm thời = g(n) hiện tại + trọng số cạnh
            tentative_g_score = g_score[current_node] + weight

            if tentative_g_score < g_score[neighbor]:
                # Đây là đường đi tốt nhất đã tìm thấy
                came_from[neighbor] = current_node
                g_score[neighbor] = tentative_g_score
                f_score = tentative_g_score + heuristics.get(neighbor, 0)
                heapq.heappush(open_set, (f_score, neighbor))

    return None, float('inf')

def reconstruct_path(came_from, current):
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    return path[::-1]

# --- Dữ liệu thử nghiệm ---
# Đồ thị ví dụ: Node kề và trọng số cạnh (chi phí g)
example_graph = {
    'A': [('B', 1), ('C', 4)],
    'B': [('A', 1), ('C', 2), ('D', 5)],
    'C': [('A', 4), ('B', 2), ('D', 1)],
    'D': [('B', 5), ('C', 1)]
}

# Giá trị Heuristic (h) từ mỗi node đến đích 'D'
example_heuristics = {
    'A': 7,
    'B': 6,
    'C': 1,
    'D': 0
}

# Chạy thử
start_node = 'A'
goal_node = 'D'
path, total_cost = a_star_search(example_graph, example_heuristics, start_node, goal_node)

if path:
    print(f"Đường đi ngắn nhất từ {start_node} đến {goal_node}: {' -> '.join(path)}")
    print(f"Tổng chi phí (g): {total_cost}")
else:
    print("Không tìm thấy đường đi.")

Đường đi ngắn nhất từ A đến D: A -> C -> D
Tổng chi phí (g): 5
